# ES → VA Evaluation & Dialect Analysis

Reads pre-computed hypotheses from `eval_results_1k.xlsx` — **no inference needed**.

**Models evaluated:** baseline, sft, grpo (v1), grpo2 (v2)

**Analyses:**
1. Summary of pre-computed metrics (chrF, BLEU, TER, BLEURT, COMET, composite)
2. Dialectal VA score — proportion of VA-specific morphological forms per model
3. Per-feature breakdown table (30 CA↔VA contrastive pairs)
4. Bar chart per dialectal feature
5. Radar / multi-dimensional comparison
6. Qualitative examples where models diverge


---
## 1. Install & Import

In [ ]:
%%capture
!pip install -q openpyxl pandas matplotlib seaborn numpy

In [ ]:
import re, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

MODELS = ["baseline", "sft", "grpo", "grpo2"]
COLORS = {"baseline":"#6B7280","sft":"#10B981","grpo":"#6366F1","grpo2":"#F59E0B"}
LABELS = {"baseline":"Baseline","sft":"SFT","grpo":"GRPO v1","grpo2":"GRPO v2"}

plt.rcParams.update({"font.family":"DejaVu Sans","axes.spines.top":False,"axes.spines.right":False})
print("Ready.")

---
## 2. Load Data from Excel

In [ ]:
# ← adjust path to your Excel file
XLSX_PATH = "eval_results_1k.xlsx"

dfs = {}
for model in MODELS:
    dfs[model] = pd.read_excel(XLSX_PATH, sheet_name=model)
    print(f"  {model}: {len(dfs[model])} rows")

df_summary = pd.read_excel(XLSX_PATH, sheet_name="summary")
print("\nSummary:")
print(df_summary.to_string(index=False))

In [ ]:
df_ref = dfs["baseline"].sort_values("id").reset_index(drop=True)
gold_es  = df_ref["source_es"].tolist()
gold_va  = df_ref["reference_va"].tolist()

hyps = {}
for model in MODELS:
    df_m = dfs[model].sort_values("id").reset_index(drop=True)
    hyps[model] = df_m["hypothesis"].fillna("[EMPTY]").tolist()

print(f"Loaded {len(gold_es)} sentence pairs.")
print("\nExample (id=0):")
print(f"  ES  : {gold_es[0]}")
print(f"  REF : {gold_va[0]}")
for m in MODELS:
    print(f"  {m.upper():<8}: {hyps[m][0]}")

---
## 3. Pre-computed Metrics Summary

In [ ]:
METRIC_COLS = ["chrF","BLEU","TER","BLEURT","COMET","composite_reward"]

df_show = df_summary.set_index("model")[METRIC_COLS].reindex(MODELS)
print("\n" + "="*75)
print(f"  {'Model':<10}" + "".join(f"{c:>12}" for c in METRIC_COLS))
print("="*75)
for model, row in df_show.iterrows():
    print(f"  {model:<10}" + "".join(f"{row[c]:>12.4f}" for c in METRIC_COLS))
print("="*75)
print("  Note: Lower TER = better. Higher all others = better.")

---
## 4. Dialectal VA Score

Counts how often each model produces Valencian-specific forms (e.g. *esta, seua, tindre, faena*)
vs the standard Catalan equivalents (*aquesta, seva, tenir, feina*).


In [ ]:
CA_VA_FEATURES = {
    "aquesta":"esta","aquest":"este","aquestes":"estes","aquests":"estos",
    "seva":"seua","seves":"seues",
    "darrer":"últim","darrers":"últims","darrera":"última",
    "tenir":"tindre","obtenir":"obtindre","veure":"vore",
    "segueix":"seguix","segueixen":"seguixen","requereix":"requerix",
    "divideix":"dividix","constitueixen":"constituïxen","absorbeixen":"absorbixen",
    "nens":"xiquets","nen":"xiquet","nena":"xiqueta","nenes":"xiquetes",
    "feina":"faena",
}

def dialectal_score(hypotheses, label):
    valid  = [h.lower() for h in hypotheses if h not in ("[SKIPPED]","[EMPTY]",None)]
    corpus = " ".join(valid)
    per_feature = {}
    total_va = total_ca = 0
    for ca, va in CA_VA_FEATURES.items():
        va_hits = len(re.findall(r"\b" + re.escape(va) + r"\b", corpus))
        ca_hits = len(re.findall(r"\b" + re.escape(ca) + r"\b", corpus))
        total   = va_hits + ca_hits
        per_feature[ca] = {"va_form":va,"va_hits":va_hits,"ca_hits":ca_hits,
                           "va_rate":va_hits/total if total>0 else None}
        total_va += va_hits; total_ca += ca_hits
    overall = total_va/(total_va+total_ca) if (total_va+total_ca)>0 else 0.0
    print(f"  {label:<10} Dialectal VA score: {overall:.1%}  ({total_va} VA / {total_va+total_ca} slots)")
    return overall, per_feature

scores, feats = {}, {}
for m in MODELS:
    scores[m], feats[m] = dialectal_score(hyps[m], m)

---
## 5. Per-feature Breakdown Table

In [ ]:
rows = []
for ca, va in CA_VA_FEATURES.items():
    row = {"CA form":ca,"VA form":va}
    for m in MODELS:
        r = feats[m][ca]["va_rate"]
        row[f"{m} VA%"]  = f"{r:.0%}" if r is not None else "—"
        row[f"{m} va/ca"] = f"{feats[m][ca]['va_hits']}/{feats[m][ca]['ca_hits']}"
    rows.append(row)

df_feat = pd.DataFrame(rows).sort_values("CA form")
pd.set_option("display.max_rows",60); pd.set_option("display.max_columns",20)
print(df_feat.to_string(index=False))

---
## 6. Bar Chart — Dialectal VA Score by Model

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
vals     = [scores[m]*100 for m in MODELS]
best_val = max(vals)
bars     = ax.bar(range(len(MODELS)), vals, color=[COLORS[m] for m in MODELS],
                  width=0.55, zorder=3, edgecolor="white")
for bar, m, v in zip(bars, MODELS, vals):
    bar.set_alpha(1.0 if v==best_val else 0.72)
    ax.text(bar.get_x()+bar.get_width()/2, v+0.8, f"{v:.1f}%",
            ha="center", va="bottom", fontsize=10,
            fontweight="bold" if v==best_val else "normal")
ax.set_xticks(range(len(MODELS)))
ax.set_xticklabels([LABELS[m] for m in MODELS], fontsize=10)
ax.set_ylabel("Valencian Form Usage Rate (%)", fontsize=10)
ax.set_title("Dialectal Valencian Score", fontsize=12, fontweight="bold")
ax.set_ylim(0, max(vals)*1.2)
ax.grid(True, alpha=0.3, linestyle="--")
plt.tight_layout()
plt.savefig("fig_dialectal_score.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: fig_dialectal_score.png")

---
## 7. Bar Charts — All Translation Metrics

In [ ]:
data = {}
for m in MODELS:
    row = df_summary[df_summary["model"]==m].iloc[0].to_dict()
    row["Dialectal_VA"] = scores[m]
    data[m] = row

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, metric, ylabel, best_high in zip(axes,
        ["chrF","BLEU","TER"], ["chrF Score","BLEU Score","TER (↓ better)"],
        [True, True, False]):
    vals     = [data[m][metric] for m in MODELS]
    best_val = min(vals) if not best_high else max(vals)
    bars     = ax.bar(range(len(MODELS)), vals, color=[COLORS[m] for m in MODELS],
                      width=0.6, zorder=3, edgecolor="white")
    for bar, m, v in zip(bars, MODELS, vals):
        bar.set_alpha(1.0 if v==best_val else 0.72)
        ax.text(bar.get_x()+bar.get_width()/2, v+(0.3 if best_high else 0.3), f"{v:.1f}",
                ha="center", va="bottom", fontsize=9,
                fontweight="bold" if v==best_val else "normal")
    ax.set_xticks(range(len(MODELS)))
    ax.set_xticklabels([LABELS[m] for m in MODELS], fontsize=9)
    ax.set_ylabel(ylabel, fontsize=10); ax.set_title(metric, fontsize=12, fontweight="bold")
    ax.set_ylim(min(vals)*0.88, max(vals)*1.12)
    ax.grid(True, alpha=0.3, linestyle="--")

plt.suptitle("Translation Quality — ES → Valencian (1k test sentences)", fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig_translation_metrics.png", dpi=300, bbox_inches="tight")
plt.show()

---
## 8. Radar Chart — Multi-dimensional Profile

In [ ]:
categories = ["chrF","BLEU","TER↓","BLEURT","COMET","Dialectal VA"]
N = len(categories)
ranges = {"chrF":(60,90),"BLEU":(35,68),"TER↓":(18,44),"BLEURT":(0.2,0.6),
          "COMET":(0.89,0.945),"Dialectal VA":(0,0.45)}
raw_keys = ["chrF","BLEU","TER","BLEURT","COMET","Dialectal_VA"]

norm_data = {}
for m in MODELS:
    vals_norm = []
    for cat, rk in zip(categories, raw_keys):
        lo, hi = ranges[cat]
        v = data[m][rk]
        v_norm = (1-(v-lo)/(hi-lo)) if cat=="TER↓" else (v-lo)/(hi-lo)
        vals_norm.append(max(0,min(1,v_norm)))
    norm_data[m] = vals_norm

angles = [n/float(N)*2*np.pi for n in range(N)] + [0]
fig, ax = plt.subplots(figsize=(7,7), subplot_kw=dict(polar=True))
for m in MODELS:
    vals = norm_data[m] + [norm_data[m][0]]
    ax.plot(angles, vals, color=COLORS[m], lw=2.2, label=LABELS[m])
    ax.fill(angles, vals, color=COLORS[m], alpha=0.1)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(categories, size=10, fontweight="bold")
ax.set_ylim(0,1); ax.set_yticks([0.25,0.5,0.75,1.0]); ax.set_yticklabels([".25",".5",".75","1.0"],size=7,color="grey")
ax.set_title("Multi-dimensional Evaluation Profile", size=13, fontweight="bold", pad=20)
patches = [mpatches.Patch(color=COLORS[m],label=LABELS[m]) for m in MODELS]
ax.legend(handles=patches, loc="upper right", bbox_to_anchor=(1.35,1.15), fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("fig_radar.png", dpi=300, bbox_inches="tight")
plt.show()

---
## 9. Qualitative Examples

In [ ]:
def contains_ca_marker(text):
    t = text.lower()
    return any(re.search(r"\b"+re.escape(ca)+r"\b", t) for ca in CA_VA_FEATURES)

interesting = [
    i for i in range(len(gold_va))
    if contains_ca_marker(hyps["baseline"][i])
    and any(not contains_ca_marker(hyps[m][i]) for m in ["sft","grpo","grpo2"])
][:10]

print(f"Found {len(interesting)} sentences where baseline uses CA forms but fine-tuned models correct them.\n")
print("="*100)
for i in interesting[:5]:
    print(f"ID   : {i}")
    print(f"ES   : {gold_es[i]}")
    print(f"REF  : {gold_va[i]}")
    for m in MODELS:
        print(f"{m.upper():<8}: {hyps[m][i]}")
    print("-"*100)

---
## 10. Export Final Summary

In [ ]:
# Add dialectal scores to summary
df_out = df_summary.copy()
df_out["Dialectal_VA"] = df_out["model"].map(scores)
df_out.to_csv("summary_metrics_full.csv", index=False)
print("Saved: summary_metrics_full.csv")
print(df_out.to_string(index=False))

# Export JSON
output = {
    "dataset"  : "gplsi/ES-VA_translation_test",
    "task"     : "ES → VA translation",
    "n_total"  : len(gold_es),
    "metrics"  : {m: {**{k:round(float(v),4) if isinstance(v,(int,float)) else v
                         for k,v in df_summary[df_summary["model"]==m].iloc[0].to_dict().items()},
                      "Dialectal_VA":scores[m]} for m in MODELS},
}
with open("eval_es_va_summary.json","w",encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
print("Saved: eval_es_va_summary.json")